<a href="https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

**The queue**: sort test-set pages by the grouped-split Logistic Regression's predicted
probability of answered_away (descending). The exact order of the top 50 rows matters more than
anything else here — if an editor can only realistically review ~50 pages in a cycle,
precision@50=0.58 (Week 6) IS the number that determines whether this queue is useful, not the
overall F1=0.350, which is why this section optimizes the top of the list rather than the whole
ranking.

**Rules provide the baseline logic, the model provides the sort order** — this queue doesn't
throw away the audited rule from Week 4 (`pattern_group`), it uses the model's predicted
probability to rank and cross-check what the rule already labeled. That's more defensible than
either piece alone: the rule has no way to rank within a label, and the model alone would be a
black-box order with no rule-based sanity check on it.

**Archetype → action mapping, four actions**:
- **structural_fix** — top 50 by predicted probability, rule-labeled answered_away, model
  confidence medium or high (predicted prob >= 0.3), AND no extreme swing flagged. Add or improve
  a direct-answer block (FAQ, clear summary near the top) aimed at the query.
- **routine_refresh** — rule-labeled normal_decay (both impressions and clicks down), no extreme
  swing. Standard content freshening: update stats, re-check target keywords, improve on-page
  relevance.
- **verify_then_review** — impr_change_pct or click_change_pct moved more than 300% in either
  direction. A swing that large is as likely to be a tracking artifact (a GA4 property change, a
  GSC verification hiccup) as a real behavioral shift, so it gets checked before it gets acted
  on — this applies even to rows that would otherwise qualify as structural_fix.
- **investigate_quiet_risk** — model confidence high (predicted prob >= 0.6), no extreme swing,
  but the rule did NOT label the page answered_away. This is the queue's most important guardrail
  row type: the model is seeing something the hand-written rule's thresholds miss, and that
  disagreement is itself the signal worth a human's attention.

**Reason codes** on every row: `impr_stable_clicks_down`, `both_declining`, `no_clear_pattern`,
`large_swing_flag` (routes to verify_then_review), `rule_model_disagreement` (routes to
investigate_quiet_risk).

**Guardrail — excluded from the queue entirely, not just down-ranked**: pages with fewer than 60
days of history before the March 2026 decision date (1,268 pages excluded on this run). A
brand-new page's prior30 impressions and clicks are naturally low and volatile, so its pct-change
columns can swing hugely from early-indexing noise alone.

**What the real run showed on the swing threshold**: percent-change swings turned out to be
common across this queue, not a rare edge case — the median max absolute swing (impressions or
clicks, whichever moved more) was 57%, the mean 84.6%, and the top end reaches into the
thousands of percent (max 11,647%), which happens with small prior-period denominators. At a
100% threshold, 11 of the top 50 ranked rows got intercepted by the swing check before they
could register as structural_fix, leaving only 26 structural_fix rows out of a 50-row top pool.
Loosening the threshold to 300% (checked directly against the distribution: 4 of the top 50 rows
still exceed it, versus 11 at 100%) raised structural_fix to 28 of the top 50 on the actual run.
The remaining 22 rows split as 18 investigate_quiet_risk and 4 verify_then_review — meaning
nearly as many top-ranked pages are model/rule disagreements as are clean structural_fix
candidates, which says something real about this feature set: a fair share of what the model
rates most confidently isn't a page the rule-based label agrees is answered_away at all. That's
reported here as an honest queue composition, not something to round away — an editor working
this list will spend real time on investigate_quiet_risk rows, not just structural_fix ones.

In [13]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

con = duckdb.connect()

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_Token')
except Exception:
    from getpass import getpass
    hf_token = getpass('HF_Token: ')

con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

FACT = "'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'"
DIM = "'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'"

# Same query as Week 5/6 — prior30 = Feb 2026, last30 = March 2026
query = f"""
WITH prior AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS gsc_impressions_prior30,
           SUM(gsc_clicks) AS gsc_clicks_prior30,
           AVG(gsc_avg_position) AS gsc_avg_position_prior30
    FROM {FACT}
    WHERE month = '2026-02'
    GROUP BY content_hash_id, client_hash_id
),
last AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS gsc_impressions_last30,
           SUM(gsc_clicks) AS gsc_clicks_last30
    FROM {FACT}
    WHERE month = '2026-03'
    GROUP BY content_hash_id, client_hash_id
)
SELECT
    p.content_hash_id, p.client_hash_id,
    p.gsc_impressions_prior30, p.gsc_clicks_prior30, p.gsc_avg_position_prior30,
    l.gsc_impressions_last30, l.gsc_clicks_last30,
    d.word_count, d.content_type, d.main_intent
FROM prior p
JOIN last l USING (content_hash_id, client_hash_id)
JOIN {DIM} d USING (content_hash_id)
WHERE p.gsc_impressions_prior30 >= 50 AND p.gsc_clicks_prior30 >= 3
"""

df = con.sql(query).df()

# Rebuild pattern_group label exactly as in Week 3/5/6
df['impr_change_pct'] = (
    (df['gsc_impressions_last30'] - df['gsc_impressions_prior30'])
    / df['gsc_impressions_prior30'] * 100
)
df['click_change_pct'] = (
    (df['gsc_clicks_last30'] - df['gsc_clicks_prior30'])
    / df['gsc_clicks_prior30'] * 100
)

def assign_pattern(row):
    if row['impr_change_pct'] >= -5 and row['click_change_pct'] <= -15:
        return 'answered_away'
    elif row['impr_change_pct'] <= -15 and row['click_change_pct'] <= -15:
        return 'normal_decay'
    else:
        return 'stable_other'

df['pattern_group'] = df.apply(assign_pattern, axis=1)
df['gsc_ctr_prior30'] = df['gsc_clicks_prior30'] / df['gsc_impressions_prior30']
y = (df['pattern_group'] == 'answered_away').astype(int)
groups = df['client_hash_id']

feature_cols_numeric = ['gsc_impressions_prior30', 'gsc_clicks_prior30',
                         'gsc_avg_position_prior30', 'word_count', 'gsc_ctr_prior30']
feature_cols_categorical = ['content_type', 'main_intent']
X = pd.get_dummies(df[feature_cols_numeric + feature_cols_categorical], dummy_na=True)
X[feature_cols_numeric] = X[feature_cols_numeric].fillna(0)

# Same grouped split as Week 6 Section 2/11 — client_hash_id grouping, random_state=42
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]
df_test = df.iloc[test_idx].copy()

scaler = StandardScaler()
X_train_grp_scaled = scaler.fit_transform(X_train_grp)
X_test_grp_scaled = scaler.transform(X_test_grp)

logreg = LogisticRegression(max_iter=1000, class_weight='balanced')
logreg.fit(X_train_grp_scaled, y_train_grp)
probs = logreg.predict_proba(X_test_grp_scaled)[:, 1]

df_test['predicted_prob_answered_away'] = probs

# Guardrail: exclude pages without at least 60 days of history before the decision date
age_query = f"""
SELECT content_hash_id,
       DATE_DIFF('day', content_created_date, DATE '2026-03-31') AS content_age_days
FROM {DIM}
"""
age_df = con.sql(age_query).df()
df_test = df_test.merge(age_df, on='content_hash_id', how='left')

too_new_mask = df_test['content_age_days'] < 60
n_excluded = int(too_new_mask.sum())
print(f"Excluding {n_excluded} pages with <60 days of history (too new to trust prior30 pct change)")

queue = df_test[~too_new_mask][['content_hash_id', 'client_hash_id', 'pattern_group',
                                 'gsc_avg_position_prior30', 'gsc_ctr_prior30',
                                 'gsc_clicks_prior30', 'gsc_clicks_last30',
                                 'impr_change_pct', 'click_change_pct',
                                 'predicted_prob_answered_away']].copy()
queue = queue.sort_values('predicted_prob_answered_away', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

# Diagnostic: check the real swing distribution before picking a threshold, rather than
# guessing a round number. This confirmed 100% was catching too much of the top-ranked pool;
# 300% was chosen from this printout, not assumed ahead of time.
swing_check = queue.copy()
swing_check['max_abs_swing'] = swing_check[['impr_change_pct', 'click_change_pct']].abs().max(axis=1)

print("\nSwing size distribution across the whole queue:")
print(swing_check['max_abs_swing'].describe())
print("\nHow many of the top 50 ranked rows exceed each candidate threshold:")
for t in [100, 200, 300, 500, 1000]:
    n_top50_over = (swing_check.head(50)['max_abs_swing'] > t).sum()
    n_total_over = (swing_check['max_abs_swing'] > t).sum()
    print(f"  >{t}%: {n_top50_over} of top 50 rows, {n_total_over} of {len(swing_check)} total rows")

# Threshold set from the diagnostic above: 300% cuts top-50 interception from 11 to 4 rows
# while still catching genuinely extreme swings (max observed: 11,647%).
SWING_THRESHOLD = 300

def assign_action(row):
    large_swing = abs(row['impr_change_pct']) > SWING_THRESHOLD or abs(row['click_change_pct']) > SWING_THRESHOLD
    high_conf = row['predicted_prob_answered_away'] >= 0.6
    if row['rank'] <= 50 and row['pattern_group'] == 'answered_away' and row['predicted_prob_answered_away'] >= 0.3:
        if large_swing:
            return 'verify_then_review'
        return 'structural_fix'
    if large_swing:
        return 'verify_then_review'
    if high_conf and row['pattern_group'] != 'answered_away':
        return 'investigate_quiet_risk'
    return 'routine_refresh'

queue['action'] = queue.apply(assign_action, axis=1)

def reason_code(row):
    if abs(row['impr_change_pct']) > SWING_THRESHOLD or abs(row['click_change_pct']) > SWING_THRESHOLD:
        return 'large_swing_flag'
    if row['action'] == 'investigate_quiet_risk':
        return 'rule_model_disagreement'
    if row['impr_change_pct'] >= -5 and row['click_change_pct'] <= -15:
        return 'impr_stable_clicks_down'
    elif row['impr_change_pct'] <= -15 and row['click_change_pct'] <= -15:
        return 'both_declining'
    else:
        return 'no_clear_pattern'

queue['reason_code'] = queue.apply(reason_code, axis=1)

def confidence_band(p):
    if p >= 0.6: return 'high'
    elif p >= 0.3: return 'medium'
    else: return 'low'

queue['model_confidence'] = queue['predicted_prob_answered_away'].apply(confidence_band)

print(f"\nSwing threshold used: >{SWING_THRESHOLD}%")
print(f"\nQueue built: {len(queue):,} pages (after excluding {n_excluded} too-new pages)")
print(f"\nAction counts:")
print(queue['action'].value_counts())
print(f"\nOf the top-50-ranked pool, how it split across actions:")
print(queue.head(50)['action'].value_counts())
print(f"\nTop 10 rows:")
print(queue[['rank', 'content_hash_id', 'pattern_group', 'predicted_prob_answered_away',
             'action', 'reason_code', 'model_confidence']].head(10).to_string(index=False))

precision_at_50 = queue.head(50)['pattern_group'].eq('answered_away').mean()
print(f"\nPrecision@50 (matches Week 6 figure): {round(precision_at_50, 3)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Excluding 1268 pages with <60 days of history (too new to trust prior30 pct change)

Swing size distribution across the whole queue:
count    14073.000000
mean        84.605916
std        187.699605
min          0.096339
25%         33.333333
50%         57.142857
75%         96.453505
max      11646.997389
Name: max_abs_swing, dtype: float64

How many of the top 50 ranked rows exceed each candidate threshold:
  >100%: 11 of top 50 rows, 2640 of 14073 total rows
  >200%: 6 of top 50 rows, 893 of 14073 total rows
  >300%: 4 of top 50 rows, 406 of 14073 total rows
  >500%: 1 of top 50 rows, 150 of 14073 total rows
  >1000%: 1 of top 50 rows, 36 of 14073 total rows

Swing threshold used: >300%

Queue built: 14,073 pages (after excluding 1268 too-new pages)

Action counts:
action
routine_refresh           11819
investigate_quiet_risk     1820
verify_then_review          406
structural_fix               28
Name: count, dtype: int64

Of the top-50-ranked pool, how it split across actions:
ac

## 2. Intended use and limits

**Intended use**: a triage aid for a content editor deciding which pages to look at first in a
weekly or monthly review cycle — not an autonomous action system. It ranks pages by predicted
likelihood of the answered_away pattern and suggests which of four next steps fits, so limited
review time goes to the pages most worth a second look, with evidence attached to the suggestion
rather than a bare score.

**Cost/value thinking** (the economics behind why this queue is worth building at all): using the
same ad-equivalent framing from Week 1 — cpc as the shadow price of the clicks a page lost, i.e.
what it would cost to rebuy that same traffic via paid search. On this run, the top 50 ranked
pages carry only \$92.26 in combined ad-equivalent value at risk, and checking the underlying cpc
distribution shows why this number needs a caveat before it's used anywhere: median cpc across
the entire 14,073-page queue is \$0.00, 78.2% of the queue has cpc exactly zero, and only 16 of the
top 50 ranked rows carry any nonzero cpc at all. The \$92.26 total is a real number, but it's a
lower bound resting on the 16 priced rows, not a representative estimate across the full 50 — the
other 34 rows may carry real, unmeasured value that this proxy simply can't see.

This is a coverage gap in the data, not evidence that the other 34 pages' traffic lacks real
value: cpc is populated per keyword only where FlyRank's keyword-research pipeline found a
matching commercial query, and a large share of this queue's pages appear to be informational
content with no matched keyword-level cpc, which zero-fills rather than going honestly
unpriced-but-valuable. The honest reading is that ad-equivalent dollar framing only prices a
minority slice of this queue, and the true value at risk across the rest is unmeasured here,
not small. A more complete cost/value pass would need either a broader cpc source, a different
proxy (e.g. weighting by clicks_lost alone, treated as a volume signal rather than a dollar one),
or an explicit acknowledgment in the paper that ad-equivalent framing understates value for
informational content — reported here as a limitation of the proxy, not adjusted to look more
favorable.

Weighed against the review-cost side: 50 pages at an assumed 15 min/page is roughly 12.5 editor-
hours. Against \$92.26 in priceable value, that's about \$7.38 per review-hour — a number that
fails its own cost/value test at face value, precisely because the dollar side is undercounting
most of the queue rather than the traffic genuinely being worth that little. The click-volume side
tells a less misleading story: clicks_lost across the top 50 has a mean of 1.86 and a max of 10
per page, so even setting dollars aside, these are real, if modest, per-page click losses worth a
look — just not ones this cpc-based proxy can price honestly for two-thirds of the queue.

**Where it stops being valid**:
- **Client scope**: trained and validated on ~30 clients from Feb-March 2026. Not checked against
  a client outside that set — applying it to a new client without re-validating is outside what
  this notebook tested.
- **Time scope**: prior30/last30 is a fixed two-month comparison, not validated on a longer
  horizon or a different season. "This page will keep declining" isn't supported — only "this
  page's prior30-to-last30 pattern looked like answered_away."
- **Precision, not certainty**: precision@50=0.58 means roughly 6 in 10 of the top 50 flagged
  pages are genuinely answered_away in this held-out test. The other 4 in 10 are a real, expected
  error rate, not a bug.
- **CPC coverage gap**: 78.2% of the queue has cpc=0, and only 16 of the top 50 rows carry a real
  nonzero cpc. Any dollar figure from this notebook should be read as a lower bound on a minority,
  priceable slice of the queue, not a representative economic estimate of the whole thing.
- **Queue composition matters too**: even inside the top 50, only 28 rows are clean
  structural_fix candidates on this run — 18 are investigate_quiet_risk (model and rule
  disagree) and 4 are verify_then_review (extreme swing, check before trusting). An editor
  reading "top 50" should not assume all 50 are ready-to-fix pages; the action column, not the
  rank alone, says what each row actually needs.
- **No causal claim**: the label is a behavioral pattern (impressions steady, clicks down), not a
  confirmed cause. This dataset has no before/after intervention data, so nothing here says a
  structural fix will recover the lost clicks.
- **Ceiling is real**: four separate attempts to push past F1=0.350 (age/freshness features,
  XGBoost, hyperparameter tuning, GA4 engagement features) all landed at roughly the same place —
  this queue's quality is close to what this feature set and data window currently support, not an
  early version waiting on a quick fix.

In [14]:
# Cost/value: ad-equivalent value at risk for the flagged queue, weighed against editor review
# time. Same shadow-price logic as Week 1 (clicks lost * cpc), applied here to the actual queue
# rather than the whole dataset. Checking the warehouse schema live first, since cpc's presence
# in dim_content hasn't been confirmed for this table — never assume warehouse columns match
# the starter CSV's data dictionary.

schema_check = con.sql(f"DESCRIBE SELECT * FROM {DIM}").df()
has_cpc = 'cpc' in schema_check['column_name'].values
print(f"'cpc' column present in warehouse dim_content: {has_cpc}")

if has_cpc:
    cpc_df = con.sql(f"SELECT content_hash_id, cpc FROM {DIM}").df()
    queue_cpc = queue.merge(cpc_df, on='content_hash_id', how='left')
    queue_cpc['clicks_lost'] = (queue_cpc['gsc_clicks_prior30'] - queue_cpc['gsc_clicks_last30']).clip(lower=0)
    queue_cpc['ad_equivalent_value_at_risk'] = queue_cpc['clicks_lost'] * queue_cpc['cpc'].fillna(0)

    top_50_value = queue_cpc.head(50)['ad_equivalent_value_at_risk'].sum()
    full_queue_value = queue_cpc['ad_equivalent_value_at_risk'].sum()
    structural_fix_value = queue_cpc[queue_cpc['action'] == 'structural_fix']['ad_equivalent_value_at_risk'].sum()

    print(f"\nAd-equivalent value at risk, top 50 ranked pages (all actions): ${top_50_value:,.2f}")
    print(f"Ad-equivalent value at risk, structural_fix rows only ({(queue_cpc['action']=='structural_fix').sum()} pages): ${structural_fix_value:,.2f}")
    print(f"Ad-equivalent value at risk, full queue ({len(queue_cpc):,} pages): ${full_queue_value:,.2f}")
    print("(Proxy for cost of rebuying this traffic via paid search — not a measured revenue")
    print("loss, since this dataset has no conversion or order-value data.)")

    # Rough review-cost comparison — editor time is a placeholder assumption, stated as such
    ASSUMED_MIN_PER_PAGE_REVIEW = 15
    review_hours_top_50 = (50 * ASSUMED_MIN_PER_PAGE_REVIEW) / 60
    print(f"\nEditor review cost, top 50 pages, at an assumed {ASSUMED_MIN_PER_PAGE_REVIEW} min/page: "
          f"{review_hours_top_50:.1f} hours")
    if review_hours_top_50 > 0:
        print(f"Value at risk per review-hour: ${top_50_value / review_hours_top_50:,.2f}")
    print("This per-page minute figure is an assumption for illustration, not a measured editor")
    print("time-and-motion study — swap in a real number if one exists before quoting this externally.")
else:
    print("\n'cpc' not found in this warehouse table — cost/value section falls back to a")
    print("volume-only framing below, since a dollar estimate isn't defensible without it.")
    queue_cpc = queue.copy()
    queue_cpc['clicks_lost'] = (queue_cpc['gsc_clicks_prior30'] - queue_cpc['gsc_clicks_last30']).clip(lower=0)
    top_50_clicks_lost = queue_cpc.head(50)['clicks_lost'].sum()
    structural_fix_clicks_lost = queue_cpc[queue_cpc['action'] == 'structural_fix']['clicks_lost'].sum()
    print(f"Total clicks lost across top 50 ranked pages: {top_50_clicks_lost:,.0f}")
    print(f"Total clicks lost across structural_fix rows only: {structural_fix_clicks_lost:,.0f}")

n_clients = df['client_hash_id'].nunique()
n_train_clients = groups.iloc[train_idx].nunique()
n_test_clients = groups.iloc[test_idx].nunique()

print(f"\nClients in this notebook's data: {n_clients} ({n_train_clients} train / {n_test_clients} test)")
print(f"Decision window: prior30 = Feb 2026, last30 = March 2026 (fixed, not re-validated on other months)")
print(f"Precision@50: {round(precision_at_50, 3)}")
print(f"Headline F1 (grouped split, from Week 6): 0.350")
print(f"Base rate of answered_away in test set: {round(y_test_grp.mean(), 3)}")
print(f"\nTop-50 action composition: {queue.head(50)['action'].value_counts().to_dict()}")
print("cpc distribution across the queue:")
print(queue_cpc['cpc'].describe())
print(f"\nNull cpc count: {queue_cpc['cpc'].isna().sum()} of {len(queue_cpc)}")

print("\nclicks_lost distribution, top 50 rows:")
print(queue_cpc.head(50)['clicks_lost'].describe())

print("\nPer-row breakdown, top 10 by rank:")
print(queue_cpc.head(10)[['rank', 'content_hash_id', 'clicks_lost', 'cpc', 'ad_equivalent_value_at_risk']].to_string(index=False))

if has_cpc:
    print(f"\ncpc coverage check:")
    print(f"  Median cpc across queue: ${queue_cpc['cpc'].median():.2f}")
    print(f"  Share of queue with cpc == 0: {(queue_cpc['cpc'] == 0).mean():.1%}")
    n_nonzero_top50 = (queue_cpc.head(50)['cpc'].fillna(0) > 0).sum()
    print(f"  Rows in top 50 with nonzero cpc: {n_nonzero_top50} of 50")
    print(f"  → the ${top_50_value:,.2f} top-50 total is a lower bound driven by {n_nonzero_top50} priced rows,")
    print(f"    not a representative estimate across all 50")

'cpc' column present in warehouse dim_content: True

Ad-equivalent value at risk, top 50 ranked pages (all actions): $92.26
Ad-equivalent value at risk, structural_fix rows only (28 pages): $72.50
Ad-equivalent value at risk, full queue (14,073 pages): $14,539.44
(Proxy for cost of rebuying this traffic via paid search — not a measured revenue
loss, since this dataset has no conversion or order-value data.)

Editor review cost, top 50 pages, at an assumed 15 min/page: 12.5 hours
Value at risk per review-hour: $7.38
This per-page minute figure is an assumption for illustration, not a measured editor
time-and-motion study — swap in a real number if one exists before quoting this externally.

Clients in this notebook's data: 30 (22 train / 8 test)
Decision window: prior30 = Feb 2026, last30 = March 2026 (fixed, not re-validated on other months)
Precision@50: 0.58
Headline F1 (grouped split, from Week 6): 0.350
Base rate of answered_away in test set: 0.245

Top-50 action composition: {'str

## 3. Human review + the no-go list

**Every row needs a human before any action is taken.** This is a ranked suggestion list, not a
dispatch queue — nothing in this notebook writes to a CMS, sends a task, or triggers an edit.

**What a person must check before acting on a flagged page**:
- Read the actual query/page content, not just the score — a `structural_fix` recommendation
  means "the pattern looks like this," not "I read the page and confirmed a missing answer
  block."
- Check for an obvious confound the model can't see: a seasonal query, a competitor's new page,
  a SERP feature change, a broken internal link — any of these can produce the same
  impressions-flat-clicks-down signature without a fixable on-page cause.
- Sanity-check `model_confidence` against `reason_code` together, not in isolation — a `low`
  confidence row with `no_clear_pattern` shouldn't get the same attention as a `high` confidence
  `impr_stable_clicks_down` row, even if both technically appear in the queue.
- Don't read a dollar figure next to a page as settled fact — with 78.2% of the queue at cpc=0,
  a $0.00 ad-equivalent value on a given row usually means "unpriced," not "worthless." Treat
  clicks_lost as the more trustworthy per-page number when cpc is zero or null.
- For `verify_then_review` rows, confirm the swing is real in the source tool (GSC/GA4) before
  treating it as a content problem at all — at a 300% threshold these are genuinely extreme
  moves, but extreme can mean "real signal" or "tracking hiccup," and this notebook can't tell
  those apart on its own.
- For `investigate_quiet_risk` rows, treat the rule/model disagreement itself as the finding —
  18 of the top 50 ranked rows on this run fall here, so this isn't a rare edge case to skim past;
  the task is figuring out which side is right for that specific page, not defaulting to trust
  either one.

**What should NOT be automated**:
- Auto-publishing any content change based on this queue's action or reason code alone.
- Auto-deprioritizing pages the model ranks low — a low rank means "not flagged as answered_away
  by this model," not "confirmed fine," since precision@50=0.58 means real misses happen in both
  directions.
- Treating a page's ad-equivalent dollar value as a go/no-go review filter — given the cpc
  coverage gap, skipping "low value" pages by dollar figure alone would systematically skip
  pages this notebook simply couldn't price, not pages confirmed unimportant.
- Using this queue for anything resembling a personnel or vendor performance judgment (e.g. which
  writer's pages "declined more") — the label is a traffic pattern, not a quality judgment, and
  the dataset was never built or validated for that purpose.
- Reviewing pages the age guardrail already excluded — if a page under 60 days old shows up
  flagged anyway, that's a bug in the exclusion logic to fix in the pipeline, not a real signal
  to manually override per page.
- Extending the action mapping, swing threshold, or cost/value framing to clients or time windows
  outside what Section 2 scoped, without re-running the leakage and validation checks from Week 6
  on the new data first.

In [15]:
# No new computation needed for this section — human-review policy, not a number.
# Printing the queue's real composition so a reviewer sees, at a glance, how much of the
# queue falls into each trust tier and action type before triaging it, rather than having
# to take the markdown's claims on faith.

print("Queue rows by action:")
print(queue['action'].value_counts())
print("\nQueue rows by model_confidence band:")
print(queue['model_confidence'].value_counts())
print("\nQueue rows by reason_code:")
print(queue['reason_code'].value_counts())

print(f"\nTop-50 action composition (what an editor actually sees first):")
print(queue.head(50)['action'].value_counts())

print(f"\nRows needing a tracking-artifact check before any content action "
      f"(verify_then_review): {(queue['action'] == 'verify_then_review').sum()}")
print(f"Rows needing a rule/model disagreement investigation "
      f"(investigate_quiet_risk): {(queue['action'] == 'investigate_quiet_risk').sum()}")
print(f"Rows with cpc=0 or null in the top 50 (dollar figure unreliable, use clicks_lost instead): "
      f"{(queue_cpc.head(50)['cpc'].fillna(0) == 0).sum()} of 50")

Queue rows by action:
action
routine_refresh           11819
investigate_quiet_risk     1820
verify_then_review          406
structural_fix               28
Name: count, dtype: int64

Queue rows by model_confidence band:
model_confidence
medium    10931
high       2655
low         487
Name: count, dtype: int64

Queue rows by reason_code:
reason_code
no_clear_pattern           7116
impr_stable_clicks_down    3550
rule_model_disagreement    1820
both_declining             1181
large_swing_flag            406
Name: count, dtype: int64

Top-50 action composition (what an editor actually sees first):
action
structural_fix            28
investigate_quiet_risk    18
verify_then_review         4
Name: count, dtype: int64

Rows needing a tracking-artifact check before any content action (verify_then_review): 406
Rows needing a rule/model disagreement investigation (investigate_quiet_risk): 1820
Rows with cpc=0 or null in the top 50 (dollar figure unreliable, use clicks_lost instead): 34 of 50


## 4. Monitoring / retrain triggers

**Signals that would mean this queue has gone stale**:
- **Precision@50 drop on a fresh month**: if a new prior30/last30 window is scored with this same
  model and precision@50 falls meaningfully below 0.58 (say, below 0.45) on a held-out check,
  that's a concrete trigger to stop trusting the ranking until re-validated.
- **Base rate shift**: answered_away's base rate was 0.245 in this run's test set. A large shift
  either direction (below 0.10 or above 0.40) on new data suggests the underlying pattern has
  changed, and a model trained on the old base rate may rank poorly on the new one.
- **structural_fix count collapsing or spiking**: this run produced exactly 28 structural_fix
  rows, all within the top 50 by construction. If a future run produces a count far outside a
  rough 15-40 range, that's ambiguous on its own — it could mean the underlying pattern genuinely
  shifted, or it could mean something broke in the pipeline (a join dropped rows, a threshold got
  edited, a data refresh changed shape) — either way it's worth a manual check before trusting
  the queue, since this notebook can't tell those two causes apart automatically.
- **New client onboarding**: any client added to the review pool outside the 30 clients this
  model was validated on should get its queue treated as unvalidated until checked against a
  holdout, given the client-diversity caveat from Week 6 (extreme size imbalance, 3 to 7,393 rows
  per client, only ~30 of 59 warehouse clients survive the volume filters).
- **Feature drift**: if `gsc_avg_position_prior30` or `gsc_ctr_prior30` distributions shift
  sharply from what training saw (e.g. a SERP layout change across many queries at once), that's
  worth a manual check even before precision@50 visibly drops — it's often the leading indicator.
- **Rule/model disagreement rate climbing**: investigate_quiet_risk sat at 1,820 of 14,073 rows
  (about 13%) on this run, including 18 of the top 50. A rising share of these rows over time,
  relative to this run's baseline, suggests the rule and the model are drifting apart on what
  "declining" looks like — worth investigating even if precision@50 hasn't moved yet, since it's
  the kind of gap that tends to widen quietly before it shows up in the headline metric.
- **CPC coverage shrinking further**: this run already has 78.2% of the queue at cpc=0. If that
  share climbs even higher on a future pull, the cost/value section becomes even less
  representative and probably needs a different proxy rather than continuing to report a
  shrinking, less-trustworthy dollar figure.

**Retrain trigger**: re-run Week 5-6's full pipeline (data pull, leakage audit, grouped split,
GroupKFold check) on a rolling basis — quarterly is reasonable given how slowly the client set and
content categories seem to shift in this data — or immediately if any signal above fires.
Retraining on the same 5 features is fine unless a monitoring signal specifically points at a
feature-level problem; if `structural_fix` count or `investigate_quiet_risk` share moves sharply,
check the pipeline for a bug before assuming the model itself needs retraining.

In [16]:
# No new computation — restating the concrete numeric thresholds above so they're
# machine-readable next to the prose, not just described in words. These get exported
# in Section 5 so the paper (and any future monitoring script) has a single source of truth.

monitoring_thresholds = {
    'precision_at_50_baseline': round(precision_at_50, 3),
    'precision_at_50_alert_below': 0.45,
    'base_rate_baseline': round(y_test_grp.mean(), 3),
    'base_rate_alert_below': 0.10,
    'base_rate_alert_above': 0.40,
    'structural_fix_count_baseline': int((queue['action'] == 'structural_fix').sum()),
    'structural_fix_count_alert_range': [15, 40],
    'investigate_quiet_risk_count_baseline': int((queue['action'] == 'investigate_quiet_risk').sum()),
    'investigate_quiet_risk_share_baseline': round((queue['action'] == 'investigate_quiet_risk').mean(), 3),
    'cpc_zero_share_baseline': round((queue_cpc['cpc'] == 0).mean(), 3),
    'validated_client_count': int(n_clients),
    'retrain_cadence': 'quarterly, or immediately on any threshold breach above'
}

for k, v in monitoring_thresholds.items():
    print(f"{k}: {v}")

precision_at_50_baseline: 0.58
precision_at_50_alert_below: 0.45
base_rate_baseline: 0.245
base_rate_alert_below: 0.1
base_rate_alert_above: 0.4
structural_fix_count_baseline: 28
structural_fix_count_alert_range: [15, 40]
investigate_quiet_risk_count_baseline: 1820
investigate_quiet_risk_share_baseline: 0.129
cpc_zero_share_baseline: 0.782
validated_client_count: 30
retrain_cadence: quarterly, or immediately on any threshold breach above


## 5. Exports for the paper

Exporting the full ranked queue to `work/outputs/` (regenerated on every run, not committed — the
CI leak-guard blocks data files) and the playbook metrics to a committed JSON, so next week's
paper has a receipt for every number quoted here: the queue composition, the swing threshold
choice, the cpc coverage gap, and the monitoring thresholds all trace back to this file rather
than to a number typed into the paper by hand.

In [17]:
import os
import json

os.makedirs('../outputs', exist_ok=True)

# Full ranked queue — regenerated each run, stays out of git by design
queue_export_cols = ['rank', 'content_hash_id', 'client_hash_id', 'pattern_group',
                      'predicted_prob_answered_away', 'action', 'reason_code',
                      'model_confidence', 'gsc_avg_position_prior30', 'gsc_ctr_prior30',
                      'impr_change_pct', 'click_change_pct']
queue[queue_export_cols].to_csv('../outputs/w07_action_queue.csv', index=False)
print(f"Exported {len(queue):,} rows to work/outputs/w07_action_queue.csv")

# Playbook metrics — committed, these are the receipts the paper cites
playbook_metrics = {
    'headline_f1_grouped_split': 0.350,
    'groupkfold_mean_f1': 0.340,
    'groupkfold_std_f1': 0.034,
    'precision_at_50': round(precision_at_50, 3),
    'base_rate_answered_away': round(y_test_grp.mean(), 3),
    'n_clients_total': int(n_clients),
    'n_clients_train': int(n_train_clients),
    'n_clients_test': int(n_test_clients),
    'queue_rows': int(len(queue)),
    'excluded_too_new_pages': int(n_excluded),
    'swing_threshold_pct': int(SWING_THRESHOLD),
    'action_counts': queue['action'].value_counts().to_dict(),
    'top_50_action_composition': queue.head(50)['action'].value_counts().to_dict(),
    'monitoring_thresholds': monitoring_thresholds
}

if has_cpc:
    playbook_metrics['ad_equivalent_value_at_risk_top_50'] = round(float(top_50_value), 2)
    playbook_metrics['ad_equivalent_value_at_risk_structural_fix_only'] = round(float(structural_fix_value), 2)
    playbook_metrics['ad_equivalent_value_at_risk_full_queue'] = round(float(full_queue_value), 2)
    playbook_metrics['cpc_zero_share'] = round((queue_cpc['cpc'] == 0).mean(), 3)
    playbook_metrics['top_50_rows_with_nonzero_cpc'] = int((queue_cpc.head(50)['cpc'].fillna(0) > 0).sum())

with open('../outputs/w07_playbook_metrics.json', 'w') as f:
    json.dump(playbook_metrics, f, indent=2)

print("\nExported work/outputs/w07_playbook_metrics.json:")
print(json.dumps(playbook_metrics, indent=2))

Exported 14,073 rows to work/outputs/w07_action_queue.csv

Exported work/outputs/w07_playbook_metrics.json:
{
  "headline_f1_grouped_split": 0.35,
  "groupkfold_mean_f1": 0.34,
  "groupkfold_std_f1": 0.034,
  "precision_at_50": 0.58,
  "base_rate_answered_away": 0.245,
  "n_clients_total": 30,
  "n_clients_train": 22,
  "n_clients_test": 8,
  "queue_rows": 14073,
  "excluded_too_new_pages": 1268,
  "swing_threshold_pct": 300,
  "action_counts": {
    "routine_refresh": 11819,
    "investigate_quiet_risk": 1820,
    "verify_then_review": 406,
    "structural_fix": 28
  },
  "top_50_action_composition": {
    "structural_fix": 28,
    "investigate_quiet_risk": 18,
    "verify_then_review": 4
  },
  "monitoring_thresholds": {
    "precision_at_50_baseline": 0.58,
    "precision_at_50_alert_below": 0.45,
    "base_rate_baseline": 0.245,
    "base_rate_alert_below": 0.1,
    "base_rate_alert_above": 0.4,
    "structural_fix_count_baseline": 28,
    "structural_fix_count_alert_range": [
    

## 5-Minute Demo Outline (Week 8 Showcase)

**Question.** Out of thousands of declining pages, which one should an editor fix first — and does it need a structural fix (a direct-answer block) or a routine content refresh? FlyRank's current system flags declining pages with hand-written rules; this asks whether a learned model can do that ranking better, in the gap where fixed rules run out.

**Method.** A client-grouped Logistic Regression trained only on prior-period signals (prior impressions, clicks, position, CTR, word count, content type, intent) — deliberately excluding anything from the last30 window or the label formula itself, confirmed clean by a leakage experiment that showed a ~0.55 F1 jump when the leaky feature was reintroduced.

**One chart.** Figure 2, the F1 comparison chart — shows the honest baseline (0.135), the leaky number (0.703) as a cautionary marker, and the real result (0.350) all side by side.

**One honest result.** F1=0.350 on a held-out client-grouped split, corroborated by 5-fold GroupKFold (mean 0.340, std 0.034) — not a lucky split. Four separate attempts to push past that (new features, XGBoost, tuning, engagement data) all landed in the same place, which is treated as a ceiling finding, not a failure.

**Recommendation.** Rank pages by predicted probability, route the top 50 into a four-action playbook (structural fix / investigate quiet risk / verify then review / routine refresh), and treat any dollar figure as a lower bound — a human reviews every row before action is taken.

## Shareable Cuts

**Social post:**
Most content-decay dashboards only ask "did clicks drop." I built a classifier that separates *why* — pages where impressions fall too (normal decay, needs a refresh) versus pages holding their impressions but losing clicks anyway (consistent with something answering the query before the click happens). Client-grouped Logistic Regression on prior-period-only signals hits F1=0.350 against a 0.229 base rate, holds up under 5-fold cross-validation, and turns into a ranked queue where the top 50 pages are right 58% of the time. Full paper + honest limitations: [link].

**Employer-facing summary:**
I built a client-grouped classifier that separates two causes of declining organic search traffic — visibility loss versus click-suppression — using FlyRank's real content performance warehouse (tens of thousands of pages across 30 clients, Google Search Console + GA4 data). It reaches F1=0.350 against a 0.229 base rate on a held-out split, corroborated by cross-validation, and converts into a ranked action queue where the top 50 recommendations are correct 58% of the time.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.